# Decision Trees vs Random Forests: Iris Classification

This notebook compares Decision Tree and Random Forest classifiers on the Iris dataset.

**Topics covered:**
1. Decision Tree training and visualization
2. Random Forest training
3. Feature importance analysis
4. Cross-validation comparison
5. Decision boundary visualization via PCA

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, accuracy_score

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load the Iris Dataset

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)

print(f"Shape: {df.shape}")
print(f"Classes: {iris.target_names}")
df.head()

In [ ]:
sns.pairplot(df, hue='species', markers=['o', 's', 'D'], palette='Set2')
plt.suptitle('Iris Feature Pairplot', y=1.02)
plt.show()

## 2. Train Decision Tree and Random Forest

In [ ]:
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
rf.fit(X_train, y_train)

print(f"Decision Tree - Train: {dt.score(X_train, y_train):.4f}, Test: {dt.score(X_test, y_test):.4f}")
print(f"Random Forest - Train: {rf.score(X_train, y_train):.4f}, Test: {rf.score(X_test, y_test):.4f}")

## 3. Visualize the Decision Tree

One of the key advantages of decision trees is interpretability. We can visualize the learned rules.

In [ ]:
plt.figure(figsize=(18, 10))
plot_tree(dt, feature_names=iris.feature_names, class_names=iris.target_names,
          filled=True, rounded=True, fontsize=10)
plt.title('Decision Tree Visualization', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Feature Importance Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, name in [(axes[0], dt, 'Decision Tree'), (axes[1], rf, 'Random Forest')]:
    importances = model.feature_importances_
    idx = np.argsort(importances)
    ax.barh(np.array(iris.feature_names)[idx], importances[idx], color='steelblue', alpha=0.8)
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'{name} Feature Importance')

plt.tight_layout()
plt.show()

## 5. Cross-Validation Comparison

We use 10-fold cross-validation to get a more robust comparison of the two models.

In [ ]:
dt_scores = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=42), X, y, cv=10)
rf_scores = cross_val_score(RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42), X, y, cv=10)

print(f"Decision Tree CV: {dt_scores.mean():.4f} +/- {dt_scores.std():.4f}")
print(f"Random Forest CV: {rf_scores.mean():.4f} +/- {rf_scores.std():.4f}")

plt.figure(figsize=(8, 5))
plt.boxplot([dt_scores, rf_scores], labels=['Decision Tree', 'Random Forest'])
plt.ylabel('Accuracy')
plt.title('10-Fold Cross-Validation Comparison')
plt.show()

## 6. Decision Boundary Visualization (PCA 2D Projection)

We reduce the 4D feature space to 2D using PCA, then plot decision boundaries for both models.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, Model, name in [(axes[0], DecisionTreeClassifier(max_depth=4, random_state=42), 'Decision Tree'),
                         (axes[1], RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42), 'Random Forest')]:
    model = Model.fit(X_pca, y)
    h = 0.02
    x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
    y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='Set2')
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='Set2', edgecolors='black', s=30)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(f'{name} Decision Boundaries')

plt.tight_layout()
plt.show()